In [1]:
# pip install -U langchain-community langchain-text-splitters langchain-huggingface langchain-chroma sentence-transformers beautifulsoup4

# Modules and dependencies

In [2]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyPDFLoader

dir_loader = DirectoryLoader(
    path="./pdf",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
)

document = dir_loader.load()
print(document[2].page_content)

C:\Users\Admin\AppData\Local\Temp\ipykernel_20864\2945901892.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader


Natural Language Processing
Natural Language Processing, commonly called NLP, is an area of artificial intelligence
focused on enabling computers to process and understand human language. NLP systems
work with text or speech and can perform tasks such as classification, translation,
summarization, and question answering.
Text must usually be transformed into a numerical representation before a machine learning
model can process it. Earlier NLP systems often used methods based on word counts, while
modern systems frequently use embeddings that represent semantic meaning.
Tokenization is the process of dividing text into smaller units called tokens. Depending on the
model, a token may represent a word, part of a word, punctuation, or another meaningful piece
of text.
Embeddings represent words, sentences, or documents as numerical vectors. Text with
similar meaning can have similar vector representations, which makes embeddings useful for
semantic search, recommendation systems, clusteri

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=[
        "\n\n",
        "\n",
        " ",
        "",
    ],
)

chunks = splitter.split_documents(document)

print(f"Total Chunks: {len(chunks)}")
print(f"First Chunk:{chunks[0].page_content}")

Total Chunks: 18
First Chunk:Deep Learning and Neural Networks
Deep learning is a specialized area of machine learning that uses artificial neural networks
with multiple layers. These networks can learn complex patterns from large amounts of data
and are widely used when traditional machine learning methods are not sufficient.
An artificial neural network is made of connected computational units called neurons.
Information moves through the network from an input layer, through one or more hidden


In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
import sentence_transformers

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    # encode_kwargs={"normalize_embeddings": True},
)

c:\Users\Admin\Desktop\Python\rag_data_ingestion\langchain_coursera\language_model\laguage_model_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:01<00:00, 147.79it/s]


In [5]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",  # Where to save data locally, remove if not necessary
)

vector_store.add_documents(documents=chunks)
print(f"Indexed {len(chunks)} chunks.")

Indexed 18 chunks.


In [6]:
# Creating/ Testing Retriever

retriever = vector_store.as_retriever(search_kwargs={"k": 3})
# print(retriever)

testing_result = retriever.invoke("What is Neural Network?")
print(testing_result[0].page_content)

Deep Learning and Neural Networks
Deep learning is a specialized area of machine learning that uses artificial neural networks
with multiple layers. These networks can learn complex patterns from large amounts of data
and are widely used when traditional machine learning methods are not sufficient.
An artificial neural network is made of connected computational units called neurons.
Information moves through the network from an input layer, through one or more hidden


chat history and conversation buffer


In [13]:
from langchain_core.chat_history import InMemoryChatMessageHistory

history = InMemoryChatMessageHistory()
history.add_ai_message("Hi")
history.add_user_message("What is the capital of France?")

print(history)

AI: Hi
Human: What is the capital of France?


In [18]:
import os
from dotenv import load_dotenv
from langchain_ollama import ChatOllama

load_dotenv()
API_KEY = os.getenv("OLLAMA_API_KEY")

if not API_KEY:
    raise ValueError("API Key Not Found")

llm = ChatOllama(
    model="gpt-oss:20b",
    base_url="https://ollama.com",
    temperature=0,
    client_kwargs={"headers": {"Authorization": f"Bearer {API_KEY}"}},
)
response = llm.invoke("Hello!Introduce yourself Briefly")
print(response.content)

Hello! I’m ChatGPT, a large language model created by OpenAI. I can help answer questions, brainstorm ideas, explain concepts, and chat about a wide range of topics. Just let me know what you need!


In [ ]:
response1 = llm.invoke(history.messages)
# print(response1)

# calling previous memory

history.add_user_message("Can you briefly tell me about that city?")
response2 = llm.invoke(history.messages)
print(f"AI: {response2.content}")

# add this AI response to the chat memory
history.add_ai_message(response2.content)

for each_conversation in history.messages:
    print(f"{each_conversation.type}:{each_conversation.content}")

AI: Paris is the capital and largest city of France, situated in the north‑central part of the country along the Seine River. It’s renowned for its rich history, iconic landmarks (Eiffel Tower, Notre‑Dame, Louvre), world‑class museums, and vibrant cultural scene. The city blends historic architecture with modern life, offering countless cafés, boutiques, and art galleries. With a population of about 2.1 million in the city proper and a metropolitan area of roughly 12 million, Paris remains a major global center for finance, fashion, and the arts.
ai:Hi
human:What is the capital of France?
human:Can you briefly tell me about that city?
human:Can you briefly tell me about that city?
human:Can you briefly tell me about that city?
human:Can you briefly tell me about that city?
human:Can you briefly tell me about that city?
ai:**Paris** is the capital of France.

Paris is a historic and cultural hub located in northern France along the Seine River. It’s known for its iconic landmarks such a